In [ ]:
mental_frame = pd.read_csv('/kaggle/input/sentiment-analysis-for-mental-health/Combined Data.csv')
mental_frame.drop(labels ='Unnamed: 0', axis = 1, inplace = True)
mental_frame.dropna(inplace = True)

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(mental_frame['status'])

xtrain,xtest,ytrain,ytest = train_test_split(mental_frame['statement'],labels, test_size = 0.2, random_state = 9)

train_dataset = Dataset.from_pandas(pd.DataFrame({'text':xtrain, 'label':ytrain}))
test_dataset = Dataset.from_pandas(pd.DataFrame({'text':xtest, 'label':ytest}))

model_path = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
def preprocess_data(corpus):
    tokenized_data = tokenizer(corpus['text'], truncation = True, padding = 'max_length', max_length = 256)
    tokenized_data['labeled'] = corpus['label']
    return tokenized_data

train_dataset = train_dataset.map(preprocess_data,batched=True)
test_dataset = test_dataset.map(preprocess_data, batched=True)

train_dataset.set_format(type='torch',columns= ['input_ids', 'attention_mask', 'label'])

batchsize = 16

train_loader = DataLoader(train_dataset,batch_size = batchsize, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=batchsize)

In [ ]:
train_labels = np.array(train_dataset['label'])
class_weights = compute_class_weight(class_weight='balanced', classes = np.unique(train_labels), y = train_labels)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_path,num_labels= len(label_encoder.classes_), ignore_mismatched_sizes = False)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

class_weights = torch.tensor(class_weights,dtype=torch.float).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 10

num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name='linear', optimizer = optimizer, num_warmup_steps = 0, num_training_steps = num_training_steps)


In [ ]:
#training and eval
for epoch in range(num_epochs):
    model.train()
    
    progress_bar = tqdm(train_loader, desc=f'Epoch: {epoch+1}')
    
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        
        loss = criterion(outputs.logits, batch['label'])
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        
        progress_bar.set_postfix({"loss": loss.item()})
    
    model.eval()
    total_eval_loss = 0
    correct_predictions = 0
    
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            loss = criterion(outputs.logits, batch["label"])
            total_eval_loss += loss.item()

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            correct_predictions += torch.sum(preds == batch["label"]).item()

    avg_eval_loss = total_eval_loss / len(test_loader)
    accuracy = correct_predictions / len(test_dataset)

    print(f"Epoch {epoch+1}: Eval Loss = {avg_eval_loss:.4f}, Accuracy = {100 * accuracy:.2f}%")

print("Training complete.")